# 03销售基本盘分析

本阶段基于02构建的订单级宽表order_base，对平台整体销售表现进行初步分析，包括订单量、GMV、客单价、支付方式、地区分布和运费结构。该阶段用于建立业务规模和收入结构认知，为后续履约时效与客户满意度分析提供背景。

## 1 读取数据

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data_clean")

order_base = pd.read_csv(DATA_DIR / "order_base.csv")

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    order_base[col] = pd.to_datetime(order_base[col], errors="coerce")

print(order_base.shape)
display(order_base.head())

(99441, 28)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delay_days,...,payment_total,payment_count,payment_installments_max,payment_type_count,main_payment_type,item_count,product_count,seller_count,price_total,freight_total
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.44,-7.11,...,38.71,3.0,1.0,2.0,voucher,1.0,1.0,1.0,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.78,-5.36,...,141.46,1.0,1.0,1.0,boleto,1.0,1.0,1.0,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.39,-17.25,...,179.12,1.0,3.0,1.0,credit_card,1.0,1.0,1.0,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.21,-12.98,...,72.20,1.0,1.0,1.0,credit_card,1.0,1.0,1.0,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.87,-9.24,...,28.62,1.0,1.0,1.0,credit_card,1.0,1.0,1.0,19.90,8.72


## 2 确认分析口径
本阶段销售分析主要基于已送达且具备支付金额、商品金额和运费信息的订单展开，避免未完成、取消、不可用订单以及关键金额字段缺失订单对GMV、客单价和运费结构分析造成干扰。

In [2]:
sales_base = order_base[
    (order_base["order_status"] == "delivered")
    & order_base["payment_total"].notna()
    & order_base["price_total"].notna()
    & order_base["freight_total"].notna()
].copy()

print("全量订单数:", order_base["order_id"].nunique())
print("已送达订单数:", order_base[order_base["order_status"] == "delivered"]["order_id"].nunique())
print("销售分析样本订单数:", sales_base["order_id"].nunique())
print(
    "销售分析样本占全量订单比例:",
    round(sales_base["order_id"].nunique() / order_base["order_id"].nunique(), 4),
)

全量订单数: 99441
已送达订单数: 96478
销售分析样本订单数: 96477
销售分析样本占全量订单比例: 0.9702


## 3 整体经营指标

In [3]:
overall_sales = pd.DataFrame({
    "metric":[
        "orders",
        "customers",
        "gmv",
        "avg_order_value",
        "avg_item_count",
        "avg_freight",
        "freight_to_product_rate",
    ],
    "value":[
        sales_base["order_id"].nunique(),
        sales_base["customer_unique_id"].nunique(),
        sales_base["payment_total"].sum(),  #本项目使用payment_total作为订单GMV口径，表示订单实际支付金额。
        sales_base["payment_total"].mean(),
        sales_base["item_count"].mean(),
        sales_base["freight_total"].mean(),
        sales_base["freight_total"].sum() / sales_base["price_total"].sum(),
    ],
})

overall_sales["value"] = overall_sales["value"].round(2)

display(overall_sales)

,metric,value
0,orders,96477.00
1,customers,93357.00
2,gmv,15422461.77
3,avg_order_value,159.86
4,avg_item_count,1.14
5,avg_freight,22.79
6,freight_to_product_rate,0.17


## 4 月度订单和GMV趋势

In [4]:
monthly_sales = (
    sales_base
    .assign(
        order_month=lambda x: x["order_purchase_timestamp"].dt.to_period("M").astype(str)
    )
    .groupby("order_month", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        gmv=("payment_total", "sum"),
        avg_order_value=("payment_total", "mean"),
    )
)

monthly_sales[["gmv", "avg_order_value"]] = monthly_sales[["gmv", "avg_order_value"]].round(2)

display(monthly_sales)

,order_month,orders,gmv,avg_order_value
0,2016-10,265,46566.71,175.72
1,2016-12,1,19.62,19.62
2,2017-01,750,127545.67,170.06
3,2017-02,1653,271298.65,164.13
4,2017-03,2546,414369.39,162.75
5,2017-04,2303,390952.18,169.76
6,2017-05,3546,567066.73,159.92
7,2017-06,3135,490225.60,156.37
8,2017-07,3872,566403.93,146.28
9,2017-08,4193,646000.61,154.07


## 5 地区销售分布

In [5]:
state_sales = (
    sales_base
    .groupby("customer_state", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        customers=("customer_unique_id", "nunique"),
        gmv=("payment_total", "sum"),
        avg_order_value=("payment_total", "mean"),
        avg_freight=("freight_total", "mean"),
    )
    .sort_values("gmv", ascending=False)
)

state_sales["gmv_share"] = (
    state_sales["gmv"] / state_sales["gmv"].sum()
).round(4)

state_sales[["gmv", "avg_order_value", "avg_freight"]] = state_sales[
    ["gmv", "avg_order_value", "avg_freight"]
].round(2)

display(state_sales.head(10))

,customer_state,orders,customers,gmv,avg_order_value,avg_freight,gmv_share
25,SP,40500,39155,5770266.19,142.48,17.33,0.3741
18,RJ,12350,11917,2055690.45,166.45,23.95,0.1333
10,MG,11354,11001,1819277.61,160.23,23.46,0.1180
22,RS,5345,5168,861802.40,161.24,24.80,0.0559
17,PR,4923,4769,781919.55,158.83,23.49,0.0507
23,SC,3546,3449,595208.40,167.85,24.85,0.0386
4,BA,3256,3158,591270.60,181.59,29.96,0.0383
6,DF,2080,2019,346146.17,166.42,23.86,0.0224
8,GO,1957,1895,334294.22,170.82,26.25,0.0217
7,ES,1995,1928,317682.65,159.24,24.57,0.0206


## 6 支付方式销售表现分析

In [6]:
payment_sales = (
    sales_base
    .groupby("main_payment_type",as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        customers=("customer_unique_id", "nunique"),
        gmv=("payment_total", "sum"),
        avg_order_value=("payment_total", "mean"),
        median_order_value=("payment_total", "median"),
        avg_installments=("payment_installments_max", "mean"),
    )
    .sort_values("gmv", ascending=False)
)

payment_sales["gmv_share"] = (
    payment_sales["gmv"] / payment_sales["gmv"].sum()
).round(4)

payment_sales[["gmv", "avg_order_value", "median_order_value", "avg_installments"]] = payment_sales[["gmv", "avg_order_value", "median_order_value", "avg_installments"]].round(2)

display(payment_sales)

,main_payment_type,orders,customers,gmv,avg_order_value,median_order_value,avg_installments,gmv_share
1,credit_card,72825,70618,12102206.16,166.18,109.41,3.55,0.7847
0,boleto,19191,18717,2769932.58,144.33,93.78,1.00,0.1796
3,voucher,2977,2907,341951.91,114.86,81.36,1.15,0.0222
2,debit_card,1484,1469,208371.12,140.41,89.62,1.00,0.0135


## 7 运费结构分析

In [7]:
sales_base["freight_to_product_rate"] = (
    sales_base["freight_total"] / sales_base["price_total"]
)

freight_summary = (
    pd.Series(
        {
            "avg_freight": sales_base["freight_total"].mean(),
            "median_freight": sales_base["freight_total"].median(),
            "avg_freight_to_product_rate": sales_base[
                "freight_to_product_rate"
            ].mean(),
            "median_freight_to_product_rate": sales_base[
                "freight_to_product_rate"
            ].median(),
        }
    )
    .round(2)
    .to_frame("value")
)

display(freight_summary)

,value
avg_freight,22.79
median_freight,17.17
avg_freight_to_product_rate,0.31
median_freight_to_product_rate,0.22


## 总结

本阶段基于order_base对已送达且关键金额字段完整的订单进行了销售基本盘分析，主要关注订单量、GMV、客单价、月度趋势、地区分布、支付方式和运费结构。

从整体指标看，销售分析样本共包含96477个订单，GMV约1542.25万，平均客单价约159.86。平均每单商品件数约1.14，说明多数订单以单件或少量商品为主；平均运费约22.79，运费相对商品金额的整体比例约0.17，说明运费在订单金额结构中具有一定影响。

从时间维度看，2016年9月至12月订单量较少，且2016年11月没有订单记录，说明数据集早期月份可能不是完整经营周期。2017年后订单量和GMV明显提升，其中2017年11月、2018年3月至5月GMV较高。后续若进行时间趋势分析，应重点关注2017年1月至2018年8月这段相对稳定的区间，避免起止边界月份影响判断。

从地区维度看，销售贡献具有明显区域集中现象。SP州贡献约577.03万GMV，占总GMV约37.41%，是平台最核心市场；RJ和MG分别贡献约13.33%和11.80%的GMV。前三个州合计贡献超过60%的GMV，说明后续履约和评分分析应重点关注这些高销售区域是否也存在延迟或差评风险。

从支付方式看，credit_card是最主要的支付方式，贡献约78.47%的GMV，平均客单价约166.18，高于boleto、voucher和debit_card。credit_card订单的平均分期数约3.55，说明信用卡及分期支付可能更多出现在较高金额订单中。但该结果仅为观察性分析，不能直接说明分期支付导致订单金额上升。

从运费结构看，整体口径下，总运费相当于总商品金额的约17%；订单口径下，平均每单运费/商品金额约31%，中位数约22%。两者存在差异，说明部分小额订单的运费占商品金额比例较高。该指标仅用于观察订单中的运费负担，不代表平台真实物流成本或利润率。

综上，Olist销售基本盘呈现出较明显的地区集中和支付方式集中：SP、RJ、MG是核心销售区域，credit_card是主要支付方式。03阶段为后续履约分析提供了业务背景；04将进一步分析这些成交订单在配送时效和客户评分上的表现，判断成交后的物流体验是否可能影响客户满意度。